# Geração dos dados: descrição matemática

Seja um conjunto fixo de pontos de entrada

$$
x = (x_1, x_2, \dots, x_n), \qquad x_i \in \mathbb{R}.
$$

---

## 1. Dataset não correlacionado (ruído i.i.d.)

Cada observação é gerada de forma independente:

$$
y_i \sim \mathcal N(0, \sigma^2), \qquad i = 1,\dots,n.
$$

Em forma vetorial:

$$
\mathbf y \sim \mathcal N(\mathbf 0, \sigma^2 I).
$$

Aqui:
- $I$ é a matriz identidade
- não há correlação entre observações
- a incerteza é puramente local (covariância diagonal)

---

## 2. Dataset correlacionado (Processo Gaussiano)

### 2.1 Kernel (estrutura de correlação)

A correlação entre os dados é especificada por um kernel SE/RBF:

$$
k(x_i, x_j)
=
\sigma_f^2
\exp\!\left(
-\frac{(x_i - x_j)^2}{2\ell^2}
\right),
$$

onde:
- $\sigma_f^2$ controla a amplitude do processo
- $\ell$ controla o comprimento de correlação

A matriz de covariância do sinal é:

$$
K_{ij} = k(x_i, x_j).
$$

---

### 2.2 Ruído independente de medição

Assumimos também um ruído i.i.d. associado às observações:

$$
\boldsymbol\eta \sim \mathcal N(\mathbf 0, \sigma_n^2 I).
$$

---

### 2.3 Covariância total dos dados observados

A covariância do vetor observado $\mathbf y$ é:

$$
\mathrm{Cov}(\mathbf y)
=
K + \sigma_n^2 I
\;\equiv\;
K_y.
$$

Isso separa claramente:
- estrutura correlacionada: $K$
- incerteza local: $\sigma_n^2 I$

---

## 3. Amostragem via decomposição de Cholesky

Como $K_y$ é simétrica e definida positiva, ela admite uma decomposição de Cholesky:

$$
K_y = L L^\top,
$$

onde $L$ é triangular inferior.

---

### 3.1 Fonte de aleatoriedade

Geramos um vetor de variáveis independentes:

$$
\boldsymbol\epsilon \sim \mathcal N(\mathbf 0, I).
$$

---

### 3.2 Geração do dado correlacionado

Definimos:

$$
\mathbf y = L \boldsymbol\epsilon.
$$

Então:

$$
\mathbb E[\mathbf y] = \mathbf 0,
\qquad
\mathrm{Cov}(\mathbf y) = L I L^\top = K_y.
$$

---

## 4. Interpretação conceitual

- A aleatoriedade fundamental vem de $\boldsymbol\epsilon$
- A correlação entre os dados é induzida pela transformação linear $L$
- O kernel define a geometria da incerteza
- O termo diagonal representa incerteza local não estruturada

Em palavras:

> **Os dados são correlacionados porque compartilham as mesmas fontes latentes de aleatoriedade, combinadas segundo a geometria definida pelo kernel.**

---

## 5. Forma equivalente (opcional)

De forma equivalente, podemos escrever:

$$
\mathbf y = \mathbf f + \boldsymbol\eta,
$$

com:

$$
\mathbf f \sim \mathcal N(\mathbf 0, K),
\qquad
\boldsymbol\eta \sim \mathcal N(\mathbf 0, \sigma_n^2 I).
$$

As duas formulações são matematicamente idênticas.

In [ ]:
import numpy as np
import ROOT

# -----------------------------
# 1) Reproducibility
# -----------------------------
rng = np.random.default_rng(7)

# -----------------------------
# 2) Inputs
# -----------------------------
n = 40
x = np.linspace(0, 10, n)

# -----------------------------
# 3) Dataset A: non-correlated (i.i.d. noise)
# -----------------------------
sigma_noise = 1.0
y_uncorr = rng.normal(0, sigma_noise, size=n)

# -----------------------------
# 4) Dataset B: correlated (SE/RBF kernel + small noise)
#    using Cholesky Decomposition
# -----------------------------
def rbf_kernel(x1, x2, sigma_f=1.5, ell=1.2):
    x1 = np.asarray(x1)[:, None]
    x2 = np.asarray(x2)[None, :]
    sqdist = (x1 - x2) ** 2
    return (sigma_f**2) * np.exp(-0.5 * sqdist / (ell**2))

sigma_f = 1.5
ell = 1.2
sigma_n = 0.2  # small i.i.d. noise std

K = rbf_kernel(x, x, sigma_f=sigma_f, ell=ell)

# Option A (most common): treat noise as part of the covariance
K_y = K + (sigma_n**2) * np.eye(n)

# Numerical jitter (helps Cholesky if K_y is nearly singular)
K_y += 1e-10 * np.eye(n)

# Cholesky factor: K_y = L L^T
L = np.linalg.cholesky(K_y)

# White noise epsilon ~ N(0, I)
eps = rng.normal(0, 1.0, size=n)

# Correlated sample y = L eps  => y ~ N(0, K_y)
y_corr_noisy = L @ eps

# (Optional) If you also want the latent "signal" f separately:
#   f ~ N(0, K),  noise ~ N(0, sigma_n^2 I),  y = f + noise
# Lf = np.linalg.cholesky(K + 1e-10*np.eye(n))
# f = Lf @ rng.normal(0, 1.0, size=n)
# y_corr_noisy = f + rng.normal(0, sigma_n, size=n)

# -----------------------------
# 5) ROOT plots side-by-side
# -----------------------------
ROOT.gStyle.SetOptStat(0)

c = ROOT.TCanvas("c", "Correlated vs Non-correlated", 1100, 450)
c.Divide(2, 1)

# Helper to make a TGraph from numpy arrays
def make_graph(xarr, yarr, name):
    g = ROOT.TGraph(len(xarr), xarr.astype(np.float64), yarr.astype(np.float64))
    g.SetName(name)
    g.SetMarkerStyle(20)
    g.SetMarkerSize(1.0)
    return g

# Left: non-correlated
c.cd(1)
g1 = make_graph(x, y_uncorr, "g_uncorr")
g1.SetTitle("Dataset A: non-correlated (i.i.d. noise);x;y")
g1.Draw("AP")

# Right: correlated
c.cd(2)
g2 = make_graph(x, y_corr_noisy, "g_corr")
g2.SetTitle("Dataset B: correlated (GP sample + small noise);x;y")
g2.Draw("AP")

c.Update()

# Keep window open if running as a script
input("Press Enter to exit...")

/opt/homebrew/Cellar/root/6.38.00/lib/root/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /Users/chanayo/.pyenv/versions/3.14.0/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "


''